# M5 Dataset EDA
Exploratory analysis of the Walmart M5 daily sales dataset used for PulsePredict.

30 490 item-store series · 1 941 daily time steps · free Kaggle download

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
DATA_DIR = Path('../data/raw/m5')

In [ ]:
# Load M5 or generate synthetic data for smoke
try:
    sales = pd.read_csv(DATA_DIR / 'sales_train_evaluation.csv')
    cal = pd.read_csv(DATA_DIR / 'calendar.csv')
    day_cols = [c for c in sales.columns if c.startswith('d_')]
    row = sales.iloc[0]
    y = row[day_cols].values.astype(float)
    dates = pd.to_datetime(cal['date'].values[:len(y)])
    series_id = row['id']
    print(f'Loaded M5: {sales.shape[0]} series x {len(day_cols)} days')
except FileNotFoundError:
    print('M5 data not found — using synthetic series')
    dates = pd.date_range('2012-01-29', periods=1941)
    rng = np.random.default_rng(0)
    trend = np.linspace(10, 20, 1941)
    seasonal = 5 * np.sin(2 * np.pi * np.arange(1941) / 7)
    y = np.clip(trend + seasonal + rng.poisson(3, 1941), 0, None)
    series_id = 'SYNTHETIC_ITEM_001'

df = pd.DataFrame({'ds': dates, 'y': y})
df.tail()

In [ ]:
# Basic statistics
print(f'Series: {series_id}')
print(f'Length: {len(df)} days ({df.ds.min().date()} to {df.ds.max().date()})')
print(f'\ny statistics:')
print(df['y'].describe())
print(f'Zero fraction: {(df.y == 0).mean():.1%}')

In [ ]:
# Time series plot with yearly decomposition
fig, axes = plt.subplots(3, 1, figsize=(14, 9))

axes[0].plot(df.ds, df.y, lw=0.8, color='steelblue')
axes[0].set_title(f'Daily Sales — {series_id}', fontsize=12)
axes[0].set_ylabel('Units')
axes[0].xaxis.set_major_locator(mdates.YearLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Weekly pattern
weekly = df.copy()
weekly['dow'] = pd.to_datetime(weekly.ds).dt.dayofweek
axes[1].bar(weekly.groupby('dow')['y'].mean().index,
            weekly.groupby('dow')['y'].mean().values, color='coral')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
axes[1].set_title('Average by Day-of-Week')

# Monthly pattern
monthly = df.copy()
monthly['month'] = pd.to_datetime(monthly.ds).dt.month
axes[2].bar(monthly.groupby('month')['y'].mean().index,
            monthly.groupby('month')['y'].mean().values, color='mediumseagreen')
axes[2].set_xticks(range(1,13))
axes[2].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[2].set_title('Average by Month')

plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation (ACF / PACF)
try:
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(df['y'].fillna(0), lags=56, ax=ax1, title='ACF (56 lags)')
    plot_pacf(df['y'].fillna(0), lags=28, ax=ax2, title='PACF (28 lags)')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('statsmodels not installed — skipping ACF/PACF')

## Key Observations
- **Weekly seasonality**: ACF spikes at lags 7, 14, 21, 28 → SeasonalNaive(s=7) is a strong baseline
- **Zero-inflation**: many bottom-level M5 series have >50% zero days (slow-moving items)
- **Non-stationarity**: upward trend in aggregate; some store-level series plateau or decline
- **Holiday spikes**: Super Bowl, Thanksgiving, Christmas create irregular demand

→ Models must handle intermittency, trend, and calendar effects.